# 99 — Build virtual T1 shot gathers and MiniSEED products (SAFE)

This notebook executes notebook 98's integration plan.

For every ordered virtual source position it:

1. loads all approved nodal-stack and original Geode/streamer products;
2. applies one constant timing shift per product;
3. resamples onto a common relative-time grid;
4. forms the union of observed receiver positions;
5. combines duplicate receivers **within the same receiver family**;
6. writes one MiniSEED file per shot and component;
7. writes an optional all-shot MiniSEED stream;
8. writes complete shot, trace, and contribution catalogs.

Geode and nodal traces are not amplitude-averaged together, even if their
coordinates coincide. Their receiver-family provenance remains explicit.

## 1. Configuration

In [1]:
from pathlib import Path
from collections import defaultdict
import json
import math

import numpy as np
import pandas as pd

from obspy import read, Stream, Trace, UTCDateTime

PROJECT_ROOT = Path('/Volumes/tachyon/LBSSP_DATA')
PLAN_ROOT = PROJECT_ROOT / '98_virtual_T1_integration_plan'
OUT_ROOT = PROJECT_ROOT / '99_virtual_T1_shot_gathers'
MSEED_ROOT = OUT_ROOT / 'per_shot_mseed'
OUT_ROOT.mkdir(parents=True, exist_ok=True)
MSEED_ROOT.mkdir(parents=True, exist_ok=True)

SHOT_CATALOG_PATH = PLAN_ROOT / '98_virtual_T1_shot_catalog.csv'
PRODUCT_PLAN_PATH = PLAN_ROOT / '98_virtual_T1_product_plan.csv'

COMPONENT = 'Z'
TARGET_SAMPLING_RATE_HZ = 1000.0
OUTPUT_START_S = 0.0
OUTPUT_END_S = 1.25

RECEIVER_MATCH_TOLERANCE_M = 0.25

# Duplicate traces are combined only within a receiver family.
COMBINE_DUPLICATE_RECEIVERS_WITHIN_FAMILY = True

# Nodal stack products are assumed to represent means of their accepted source
# events. Their combining weight is n_accepted_members when available.
NODAL_WEIGHT_COLUMN = 'n_accepted_members'
RAW_GEODE_WEIGHT = 1.0

# Deterministic absolute timestamps are needed for MiniSEED. The scientific
# time axis remains relative and is recorded in the sidecar catalogs.
VIRTUAL_EPOCH = UTCDateTime(2000, 1, 1)
SHOT_TIME_STRIDE_S = 10.0

WRITE_ALL_SHOTS_MSEED = True
MSEED_ENCODING = 'FLOAT32'

pd.set_option('display.max_columns', 300)
pd.set_option('display.width', 280)

print('Output:', OUT_ROOT)
print('Target sampling rate:', TARGET_SAMPLING_RATE_HZ)
print('Output relative window:', OUTPUT_START_S, 'to', OUTPUT_END_S, 's')

Output: /Volumes/tachyon/LBSSP_DATA/99_virtual_T1_shot_gathers
Target sampling rate: 1000.0
Output relative window: 0.0 to 1.25 s


## 2. Load integration plan

In [2]:
for path in [SHOT_CATALOG_PATH, PRODUCT_PLAN_PATH]:
    if not path.exists():
        raise FileNotFoundError(f'Missing notebook-98 output: {path}')

shots = pd.read_csv(SHOT_CATALOG_PATH, low_memory=False)
products = pd.read_csv(PRODUCT_PLAN_PATH, low_memory=False)

products['include_in_virtual_shot'] = (
    products.include_in_virtual_shot.astype(str)
    .str.lower().isin(['true', '1', 'yes'])
)
products['time_shift_to_canonical_s'] = pd.to_numeric(
    products.time_shift_to_canonical_s, errors='coerce'
)
products['source_x_m'] = pd.to_numeric(
    products.source_x_m, errors='coerce'
)

products = products.loc[
    products.include_in_virtual_shot
    & products.component.astype(str).str.upper().eq(COMPONENT)
].copy()

print('Virtual shots:', len(shots))
print('Included products:', len(products))
display(
    products.groupby(
        ['product_kind', 'receiver_family', 'survey'],
        dropna=False,
    ).size().reset_index(name='n_products')
)

Virtual shots: 159
Included products: 199


,product_kind,receiver_family,survey,n_products
0,nodal_stack,nodal,T1_1m_refraction,39
1,nodal_stack,nodal,T1_2m_refraction,36
2,nodal_stack,nodal,T1_streamer_masw,80
3,nodal_stack,nodal,NaN,44


## 3. Waveform and geometry helpers

In [3]:
def normalize_component(value):
    text = str(value).strip().upper()
    return text[-1] if text and text[-1] in 'ZNE' else text


def trace_receiver_x_m(trace):
    candidates = [
        getattr(trace.stats, 'receiver_x_m', np.nan),
        getattr(trace.stats, 'distance', np.nan),
    ]

    seg2 = getattr(trace.stats, 'seg2', None)
    if seg2 is not None:
        for key in [
            'RECEIVER_LOCATION',
            'RECEIVER_STATION_NUMBER',
            'CHANNEL_NUMBER',
        ]:
            try:
                candidates.append(seg2.get(key, np.nan))
            except Exception:
                pass

    for candidate in candidates:
        value = pd.to_numeric(candidate, errors='coerce')
        if pd.notna(value):
            return float(value)

    try:
        return int(str(trace.stats.station)) / 100.0
    except Exception:
        return np.nan


def read_product_stream(product):
    path = Path(str(product.waveform_path))
    if not path.exists():
        raise FileNotFoundError(path)

    stream = read(str(path))
    selected = Stream(
        trace.copy()
        for trace in stream
        if (
            product.product_kind == 'geode_raw'
            or normalize_component(trace.stats.channel) == COMPONENT
        )
    )

    if product.product_kind == 'geode_raw':
        # SEG-2 Geode data are vertical-only in this workflow.
        for trace in selected:
            trace.stats.channel = 'GHZ'

    return selected


def assign_receiver_positions(stream, product):
    output = []

    fallback_first = pd.to_numeric(
        product.receiver_first_x_m_fallback,
        errors='coerce',
    )
    fallback_dx = pd.to_numeric(
        product.receiver_dx_m_fallback,
        errors='coerce',
    )
    reverse = str(product.reverse_trace_order_fallback).lower() in {
        'true', '1', 'yes'
    }

    trace_indices = list(range(len(stream)))
    if reverse:
        trace_indices = list(reversed(trace_indices))

    for output_index, original_index in enumerate(trace_indices):
        trace = stream[original_index].copy()
        receiver_x = trace_receiver_x_m(trace)

        # Raw Geode SEG-2 headers frequently do not carry the local line x.
        if product.product_kind == 'geode_raw' and (
            not np.isfinite(receiver_x)
            or receiver_x < -1000
            or receiver_x > 10000
        ):
            receiver_x = float(
                fallback_first + output_index * fallback_dx
            )

        if not np.isfinite(receiver_x):
            continue

        output.append((float(receiver_x), trace, original_index))

    return output


def product_weight(product):
    if product.product_kind == 'nodal_stack':
        value = pd.to_numeric(
            getattr(product, NODAL_WEIGHT_COLUMN, np.nan),
            errors='coerce',
        )
        return float(value) if pd.notna(value) and value > 0 else 1.0
    return RAW_GEODE_WEIGHT


def shifted_resampled_data(trace, time_shift_s):
    working = trace.copy()
    working.detrend('demean')

    target_rate = float(TARGET_SAMPLING_RATE_HZ)
    if not np.isclose(working.stats.sampling_rate, target_rate):
        working.interpolate(
            sampling_rate=target_rate,
            method='lanczos',
            a=12,
        )

    input_times = np.arange(working.stats.npts) / target_rate
    output_times = np.arange(
        OUTPUT_START_S,
        OUTPUT_END_S,
        1.0 / target_rate,
    )

    # A positive time_shift moves the waveform later:
    # output(t) = input(t - time_shift).
    sample_times = output_times - float(time_shift_s)

    data = np.interp(
        sample_times,
        input_times,
        np.asarray(working.data, dtype=float),
        left=np.nan,
        right=np.nan,
    )

    return output_times, data


def receiver_cluster_key(receiver_x_m):
    return int(round(float(receiver_x_m) / RECEIVER_MATCH_TOLERANCE_M))


def short_station_code(family, receiver_x_m, duplicate_index=0):
    prefix = 'N' if family == 'nodal' else 'G'
    position_code = int(round(receiver_x_m * 10))
    suffix = '' if duplicate_index == 0 else str(duplicate_index)
    return f'{prefix}{position_code:04d}{suffix}'[:5]

## 4. Build every virtual shot

In [4]:
shot_rows = []
trace_rows = []
contribution_rows = []
all_shot_stream = Stream()

for shot in shots.sort_values('virtual_shot_number').itertuples(index=False):
    shot_products = products.loc[
        products.virtual_shot_number.eq(shot.virtual_shot_number)
    ].sort_values(
        ['receiver_family', 'priority', 'product_id'],
        kind='stable',
    )

    candidate_rows = []
    product_errors = []

    for product in shot_products.itertuples(index=False):
        try:
            stream = read_product_stream(product)
            receiver_traces = assign_receiver_positions(stream, product)
            weight = product_weight(product)

            for receiver_x, trace, original_trace_index in receiver_traces:
                output_times, data = shifted_resampled_data(
                    trace,
                    product.time_shift_to_canonical_s,
                )
                candidate_rows.append({
                    'receiver_family': product.receiver_family,
                    'receiver_x_m': receiver_x,
                    'receiver_cluster_key': receiver_cluster_key(receiver_x),
                    'product_id': product.product_id,
                    'product_kind': product.product_kind,
                    'survey': product.survey,
                    'waveform_path': product.waveform_path,
                    'time_shift_s': product.time_shift_to_canonical_s,
                    'weight': weight,
                    'original_trace_index': original_trace_index,
                    'data': data,
                })

        except Exception as exc:
            product_errors.append({
                'product_id': product.product_id,
                'error': repr(exc),
            })

    candidate_frame = pd.DataFrame(candidate_rows)
    output_stream = Stream()

    if len(candidate_frame):
        grouped = candidate_frame.groupby(
            ['receiver_family', 'receiver_cluster_key'],
            sort=True,
        )

        family_position_counts = defaultdict(int)

        for (family, cluster_key), group in grouped:
            receiver_x = float(
                np.average(
                    group.receiver_x_m,
                    weights=group.weight,
                )
            )

            arrays = np.vstack(group.data.to_list())
            weights = group.weight.to_numpy(dtype=float)

            finite = np.isfinite(arrays)
            weighted_values = np.where(
                finite,
                arrays * weights[:, None],
                0.0,
            )
            weight_sum = np.where(
                finite,
                weights[:, None],
                0.0,
            ).sum(axis=0)

            combined = np.divide(
                weighted_values.sum(axis=0),
                weight_sum,
                out=np.zeros(arrays.shape[1], dtype=float),
                where=weight_sum > 0,
            )

            # If combining is disabled, retain the highest-weight contribution.
            if (
                not COMBINE_DUPLICATE_RECEIVERS_WITHIN_FAMILY
                and len(group) > 1
            ):
                chosen = group.iloc[
                    int(np.argmax(group.weight.to_numpy()))
                ]
                combined = np.nan_to_num(
                    chosen.data,
                    nan=0.0,
                )

            duplicate_index = family_position_counts[
                (family, round(receiver_x, 3))
            ]
            family_position_counts[
                (family, round(receiver_x, 3))
            ] += 1

            station = short_station_code(
                family,
                receiver_x,
                duplicate_index,
            )

            trace = Trace(
                data=np.asarray(combined, dtype=np.float32)
            )
            trace.stats.network = 'VT'
            trace.stats.station = station
            trace.stats.location = (
                'ND' if family == 'nodal' else 'GD'
            )
            trace.stats.channel = (
                'NHZ' if family == 'nodal' else 'GHZ'
            )
            trace.stats.sampling_rate = TARGET_SAMPLING_RATE_HZ
            trace.stats.starttime = (
                VIRTUAL_EPOCH
                + shot.virtual_shot_number * SHOT_TIME_STRIDE_S
                + OUTPUT_START_S
            )

            # Custom fields are useful in memory but the CSV catalogs are
            # authoritative because MiniSEED does not preserve arbitrary fields.
            trace.stats.receiver_x_m = receiver_x
            trace.stats.source_x_m = float(shot.source_x_m)
            trace.stats.virtual_shot_number = int(
                shot.virtual_shot_number
            )
            trace.stats.receiver_family = family

            output_stream += trace

            trace_number = len(output_stream)
            trace_rows.append({
                'virtual_shot_number': int(shot.virtual_shot_number),
                'virtual_shot_id': shot.virtual_shot_id,
                'source_cluster_id': shot.source_cluster_id,
                'source_x_m': float(shot.source_x_m),
                'trace_number_within_shot': trace_number,
                'receiver_family': family,
                'receiver_x_m': receiver_x,
                'station': station,
                'channel': trace.stats.channel,
                'sampling_rate_hz': TARGET_SAMPLING_RATE_HZ,
                'relative_start_s': OUTPUT_START_S,
                'relative_end_s': OUTPUT_END_S,
                'n_samples': trace.stats.npts,
                'n_contributing_products': len(group),
                'total_weight': float(group.weight.sum()),
                'contributing_product_ids': ' | '.join(
                    group.product_id.astype(str)
                ),
            })

            for contribution in group.itertuples(index=False):
                contribution_rows.append({
                    'virtual_shot_number': int(
                        shot.virtual_shot_number
                    ),
                    'virtual_shot_id': shot.virtual_shot_id,
                    'source_x_m': float(shot.source_x_m),
                    'output_receiver_family': family,
                    'output_receiver_x_m': receiver_x,
                    'product_id': contribution.product_id,
                    'product_kind': contribution.product_kind,
                    'survey': contribution.survey,
                    'input_receiver_x_m': contribution.receiver_x_m,
                    'time_shift_s': contribution.time_shift_s,
                    'weight': contribution.weight,
                    'waveform_path': contribution.waveform_path,
                    'original_trace_index': (
                        contribution.original_trace_index
                    ),
                })

    # Stable receiver ordering: x first, then family.
    output_stream.traces.sort(
        key=lambda trace: (
            float(trace.stats.receiver_x_m),
            str(trace.stats.receiver_family),
            trace.stats.station,
        )
    )

    shot_token = (
        f'{int(shot.virtual_shot_number):04d}'
        f'_x{float(shot.source_x_m):07.1f}m'
    )
    mseed_path = (
        MSEED_ROOT
        / f'T1_VIRTUAL_SHOT_{shot_token}_{COMPONENT}.mseed'
    )

    if len(output_stream):
        output_stream.write(
            str(mseed_path),
            format='MSEED',
            encoding=MSEED_ENCODING,
        )
        all_shot_stream += output_stream
        status = 'written'
    else:
        mseed_path = None
        status = 'no_output_traces'

    shot_rows.append({
        'virtual_shot_number': int(shot.virtual_shot_number),
        'virtual_shot_id': shot.virtual_shot_id,
        'source_cluster_id': shot.source_cluster_id,
        'source_x_m': float(shot.source_x_m),
        'component': COMPONENT,
        'n_planned_products': len(shot_products),
        'n_loaded_candidate_traces': len(candidate_frame),
        'n_output_traces': len(output_stream),
        'n_product_errors': len(product_errors),
        'product_errors_json': json.dumps(product_errors),
        'mseed_path': str(mseed_path) if mseed_path else None,
        'status': status,
    })

    if shot.virtual_shot_number % 20 == 0:
        print(
            f"Processed shot {shot.virtual_shot_number}/"
            f"{len(shots)} at x={shot.source_x_m:.1f} m"
        )

virtual_shot_manifest = pd.DataFrame(shot_rows)
virtual_trace_manifest = pd.DataFrame(trace_rows)
trace_contributions = pd.DataFrame(contribution_rows)

print('Written shot gathers:', int(
    virtual_shot_manifest.status.eq('written').sum()
))
print('Output traces:', len(virtual_trace_manifest))

Processed shot 20/159 at x=92.5 m
Processed shot 40/159 at x=108.0 m
Processed shot 60/159 at x=120.0 m
Processed shot 80/159 at x=132.0 m
Processed shot 100/159 at x=144.5 m
Processed shot 120/159 at x=163.0 m
Processed shot 140/159 at x=189.0 m
Written shot gathers: 159
Output traces: 5900


## 5. Write all-shot MiniSEED and catalogs

In [5]:
ALL_SHOTS_MSEED = OUT_ROOT / f'T1_virtual_all_shots_{COMPONENT}.mseed'

if WRITE_ALL_SHOTS_MSEED and len(all_shot_stream):
    all_shot_stream.traces.sort(
        key=lambda trace: (
            int(trace.stats.virtual_shot_number),
            float(trace.stats.receiver_x_m),
            str(trace.stats.receiver_family),
        )
    )
    all_shot_stream.write(
        str(ALL_SHOTS_MSEED),
        format='MSEED',
        encoding=MSEED_ENCODING,
    )
    print('Wrote:', ALL_SHOTS_MSEED)

OUTPUTS = {
    'shots': OUT_ROOT / '99_virtual_T1_shot_manifest.csv',
    'traces': OUT_ROOT / '99_virtual_T1_trace_manifest.csv',
    'contributions': OUT_ROOT / '99_virtual_T1_trace_contributions.csv',
    'summary': OUT_ROOT / '99_virtual_T1_build_summary.csv',
}

virtual_shot_manifest.to_csv(OUTPUTS['shots'], index=False)
virtual_trace_manifest.to_csv(OUTPUTS['traces'], index=False)
trace_contributions.to_csv(OUTPUTS['contributions'], index=False)

summary = pd.DataFrame([
    ('virtual_shots_planned', len(shots)),
    ('virtual_shots_written', int(
        virtual_shot_manifest.status.eq('written').sum()
    )),
    ('virtual_shots_empty', int(
        virtual_shot_manifest.status.ne('written').sum()
    )),
    ('output_traces', len(virtual_trace_manifest)),
    ('nodal_output_traces', int(
        virtual_trace_manifest.receiver_family.eq('nodal').sum()
    ) if len(virtual_trace_manifest) else 0),
    ('geode_output_traces', int(
        virtual_trace_manifest.receiver_family.eq('geode').sum()
    ) if len(virtual_trace_manifest) else 0),
    ('trace_contribution_rows', len(trace_contributions)),
], columns=['metric', 'value'])

summary.to_csv(OUTPUTS['summary'], index=False)
display(summary)

print('\nWritten:')
for name, path in OUTPUTS.items():
    print(f'  {name:14s} {path}')

Wrote: /Volumes/tachyon/LBSSP_DATA/99_virtual_T1_shot_gathers/T1_virtual_all_shots_Z.mseed


,metric,value
0,virtual_shots_planned,159
1,virtual_shots_written,159
2,virtual_shots_empty,0
3,output_traces,5900
4,nodal_output_traces,5900
5,geode_output_traces,0
6,trace_contribution_rows,6838



Written:
  shots          /Volumes/tachyon/LBSSP_DATA/99_virtual_T1_shot_gathers/99_virtual_T1_shot_manifest.csv
  traces         /Volumes/tachyon/LBSSP_DATA/99_virtual_T1_shot_gathers/99_virtual_T1_trace_manifest.csv
  contributions  /Volumes/tachyon/LBSSP_DATA/99_virtual_T1_shot_gathers/99_virtual_T1_trace_contributions.csv
  summary        /Volumes/tachyon/LBSSP_DATA/99_virtual_T1_shot_gathers/99_virtual_T1_build_summary.csv


## 6. Next step

Notebook 100 reads these ordered shot gathers and catalogs and exports:

- sparse all-shot SEG-Y, containing only observed traces;
- regularized all-shot SEG-Y, containing the full master receiver grid with
  missing combinations written as zero-valued traces marked dead;
- optional per-shot SEG-Y files.

The CSV manifests remain authoritative for receiver family and provenance.